# 01 · Data exploration and memo cleaning (Lecture 2)

**Question:** what is in the Prism data, and which memo tokens carry merchant / transaction-type cues?

Lecture 2 asked us to check the transaction, balance and link files before trusting them (row grain, joins,
null / zero / negative balances, currencies) and to clean memos conservatively: keep `memo_raw`, lowercase and
collapse spaces first, and test stronger rules one at a time. It also asked which merchants are most common per
category, verified against real memos.

Our Lecture 2 work (reviewed in Lecture 3): an accounts deep dive (28,934 transaction rows, 434 distinct memos,
20 cleaning examples) that checked the missing-currency and debit/credit assumptions. The DuckDB profile below
comes from Vanessa's PA4 notebook.

> Runs on DataHub. Cells show aggregate tables and charts only; files with memo text (errors, examples, predictions) stay in `outputs/` and are never printed here or committed.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
CONFIG = os.environ.get("CAPSTONE_CONFIG", "config.yaml")

import pandas as pd
from IPython.display import Image, display
from src.config import load_config
from src.main import run
OUT = load_config(CONFIG)["paths"]["out_dir"]
pd.set_option("display.max_colwidth", 80)

## Transactions, accounts and balances

In [ ]:
m = run(CONFIG, "explore")
summary = json.load(open(OUT / "data_summary.json"))
{k: v for k, v in summary.items() if k not in ("split_sizes", "account_files")}

In [ ]:
pd.DataFrame(summary["split_sizes"]).T

In [ ]:
summary.get("account_files", {})

In [ ]:
display(Image(OUT / "transactions_by_month.png"))
display(Image(OUT / "cumulative_transactions.png"))

The months nearest 70% and 85% of all transactions set our split cutoffs (train before April 2026,
validation April to May, test June onward). The balance file has one row per account; the link file has one row
per consumer-account pair, so joining transactions through it can duplicate activity.

## Memo cleaning (conservative)

In [ ]:
from src.clean import text_lower, text_norm, memo_pattern
examples = ["Debit: Signature purchase from TEXACO 659707401524073 TEXACO 0385201 EULESS TX US",
            "Debit: PIN purchase from WAL-MART #3392 2108 BEDFORD RD BEDFORD TXUS",
            "PURCHASE AUTHORIZED ON 07/23 NETFLIX.COM 866-579-7172 CA S464206014704870 CARD 0729"]
pd.DataFrame({"memo_raw": examples, "text_lower": [text_lower(x) for x in examples],
              "text_norm": [text_norm(x) for x in examples],
              "pattern": [memo_pattern(text_norm(x)) for x in examples]})

`text_lower` is the Lecture 2 baseline (case + spacing). `text_norm` also masks dates, card masks and
long IDs, so reference codes are not treated as vocabulary. Both are compared during model tuning (notebook 04).
The examples above are the public lecture memos, not data rows.

## Merchant frequencies per category (measured on a random sample)

In [ ]:
m = run(CONFIG, "merchants")
pd.read_csv(OUT / "rule_coverage_by_category.csv", index_col=0)

In [ ]:
top = pd.read_csv(OUT / "top_patterns_by_category.csv")
top[top.category == "Subscriptions & Entertainment"].head(15)

Pre-tag rules (`src/rules.py`) give each memo a proposed category and the matched cue. The share
with no rule match is the work left for human review and the model.